# Close-out controls — the last two open audit findings

Neither needs a GPU; Colab CPU is enough (a few minutes). This notebook closes:

**M4 — the dense-vs-SAE comparison is not capacity-matched.** Table 2 reports that decomposing ESM-2
through an InterPLM sparse autoencoder lifts close-ancestry AP from 0.534 to 0.733 at the 8M scale, and
reads that as evidence against *polysemantic superposition* (Discussion §3.1). But the dense
representation is 320-d and the SAE is 10,240-d. Part or all of the gain could simply be the 32x larger
feature budget. This notebook adds the three controls that separate those explanations:

| Representation | dim | overcomplete? | sparse/non-negative? | learned? |
|---|---|---|---|---|
| Dense ESM-2 8M (layer 4) | 320 | no | no | yes |
| **Random projection** of dense | 10,240 | yes | no | no |
| **Random ReLU projection** of dense | 10,240 | yes | yes | no |
| **Matched-nnz dense probe** | 320 | no | no | yes |
| InterPLM-8M SAE (layer 4) | 10,240 | yes | yes | **yes** |

If the SAE beats the random ReLU projection, the disentanglement claim survives. If it does not, the
effect is capacity and sparsity, not learned structure, and §3.1 must be rewritten.

**M5 — the PIC comparison was never tested.** The manuscript reports PIC r = 0.378 (p = 5.3e-3) for ESM-2
contrasts and r = 0.261 (p = 0.059) for sequence-identity contrasts, and treats that as a win. Those are
two *dependent* correlations sharing a criterion; "significant vs non-significant" is not a significant
difference. This notebook recomputes both contrasts and applies **Williams' test**.

Evaluation set throughout is the audit-corrected **strict** benchmark: 27 click-beetle positives
(4.99 effective clusters) vs **32 genuinely labelled** non-luminous negatives, cluster-weighted. The 59
unlabeled intra-genome paralogs are excluded — scoring a model for agreeing with homology-only
annotations is the circularity the paper argues against.

## 0. Setup

Upload a zip of `data/curated/dataset.tsv` and `results/scores.tsv` (that is all this needs).

In [ ]:
!pip install -q fair-esm biopython
import os, io, zipfile, numpy as np, pandas as pd, torch
from google.colab import files
if not os.path.exists("dataset.tsv"):
    up = files.upload()
    for k in up:
        if k.endswith(".zip"):
            zipfile.ZipFile(io.BytesIO(up[k])).extractall(".")
        else:
            open(k,"wb").write(up[k])
for root,_,fs in os.walk("."):
    for f in fs:
        if f in ("dataset.tsv","scores.tsv"): os.replace(os.path.join(root,f), f)
data   = pd.read_csv("dataset.tsv", sep="\t")
scores = pd.read_csv("scores.tsv",  sep="\t")
print("dataset", data.shape, "| scores", scores.shape)

## 1. Build the strict evaluation split (audit-corrected)

In [ ]:
seqs = dict(zip(data.accession, data.sequence))
S = scores.copy()
pos = S[(S.label=="positive") & (S.split_group=="click_beetle")]
neg = S[(S.split_group=="nonlum_elateroid") |
        ((S.split_group=="nonlum_other") & (S.genus=="Agrilus"))]
train_pos = S[(S.label=="positive") & (S.split_group=="firefly")]
train_neg = S[(S.label=="negative") & (S.split_group=="nonlum_other") &
              (~S.genus.isin(["Agrilus","Tenebrio","Leptinotarsa","Nicrophorus"]))]
print(f"TRAIN  {len(train_pos)} firefly positives ({train_pos.weight.sum():.2f} clusters) vs "
      f"{len(train_neg)} negatives ({train_neg.weight.sum():.2f} clusters)")
print(f"TEST   {len(pos)} click-beetle positives ({pos.weight.sum():.2f} clusters) vs "
      f"{len(neg)} labelled negatives ({neg.weight.sum():.2f} clusters)")
print(f"AP baseline (cluster-weighted prevalence) = "
      f"{pos.weight.sum()/(pos.weight.sum()+neg.weight.sum()):.4f}")

## 2. ESM-2 8M layer-4 embeddings + InterPLM-8M SAE

In [ ]:
from transformers import AutoTokenizer, EsmModel
from huggingface_hub import hf_hub_download

tok = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
esm = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D", add_pooling_layer=False).eval()
dev = "cuda" if torch.cuda.is_available() else "cpu"; esm = esm.to(dev)

sae_path = hf_hub_download("Elana/InterPLM-esm2-8m", "layer_4/ae_normalized.pt")
st = torch.load(sae_path, map_location="cpu")
W_enc, b_enc, b_pre = st["encoder.weight"], st["encoder.bias"], st["bias"]
print("SAE:", tuple(W_enc.shape), "(features x d_model)")

accs = list(S.accession)
DENSE, SAEF = {}, {}
with torch.no_grad():
    for i, a in enumerate(accs):
        t = tok(seqs[a][:1022], return_tensors="pt").to(dev)
        h = esm(**t, output_hidden_states=True).hidden_states[4][0, 1:-1].cpu()   # drop <cls>/<eos>
        DENSE[a] = h.mean(0).numpy()
        z = torch.relu((h - b_pre) @ W_enc.T + b_enc)
        SAEF[a] = z.mean(0).numpy()
        if i % 100 == 0: print(f"  {i}/{len(accs)}", end="\r")
Xd = np.stack([DENSE[a] for a in accs]); Xs = np.stack([SAEF[a] for a in accs])
print(f"\ndense {Xd.shape} | SAE {Xs.shape}")

## 3. The matched controls

Every representation is scored with the *same* pipeline: standardise, L1 logistic regression fit on
fireflies only, cluster-weighted AP on the held-out click beetles. The only thing that varies is the
representation.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

idx = {a:i for i,a in enumerate(accs)}
tr = np.array([idx[a] for a in list(train_pos.accession)+list(train_neg.accession)])
te = np.array([idx[a] for a in list(pos.accession)+list(neg.accession)])
y_tr = np.r_[np.ones(len(train_pos)), np.zeros(len(train_neg))]
y_te = np.r_[np.ones(len(pos)),       np.zeros(len(neg))]
w_tr = np.r_[train_pos.weight.values, train_neg.weight.values]
w_te = np.r_[pos.weight.values,       neg.weight.values]
BASE = pos.weight.sum()/(pos.weight.sum()+neg.weight.sum())

def evaluate(X, C=1.0):
    sc = StandardScaler().fit(X[tr])
    bw = w_tr.copy(); bw[y_tr==1] *= bw[y_tr==0].sum()/bw[y_tr==1].sum()
    lr = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=5000)
    lr.fit(sc.transform(X[tr]), y_tr, sample_weight=bw)
    s = lr.decision_function(sc.transform(X[te]))
    nnz = int((np.abs(lr.coef_[0]) > 1e-8).sum())
    return (average_precision_score(y_te, s, sample_weight=w_te),
            roc_auc_score(y_te, s, sample_weight=w_te), nnz)

rng = np.random.default_rng(0)
D = Xd.shape[1]; K = Xs.shape[1]
ap_sae, au_sae, nnz_sae = evaluate(Xs)

# random projections, averaged over 10 seeds
rp, rrp = [], []
for seed in range(10):
    g = np.random.default_rng(seed)
    P = g.normal(0, 1/np.sqrt(D), (D, K))
    rp.append(evaluate(Xd @ P)[:2])
    rrp.append(evaluate(np.maximum(Xd @ P, 0))[:2])
rp, rrp = np.array(rp), np.array(rrp)

# dense at matched sparsity: tune C so the probe uses ~nnz_sae features (capped by D=320)
best, target = None, min(nnz_sae, D)
for C in np.logspace(-3, 2, 40):
    ap, au, nnz = evaluate(Xd, C)
    if best is None or abs(nnz-target) < abs(best[2]-target): best = (ap, au, nnz)

print(f"\nAP baseline (chance) = {BASE:.4f}\n")
print(f"{'representation':38s} {'dim':>6s} {'AUROC':>7s} {'AP':>7s} {'nnz':>6s}")
print("-"*70)
ap_d, au_d, nnz_d = evaluate(Xd)
print(f"{'Dense ESM-2 8M (layer 4)':38s} {D:6d} {au_d:7.3f} {ap_d:7.3f} {nnz_d:6d}")
print(f"{'Dense, matched sparsity':38s} {D:6d} {best[1]:7.3f} {best[0]:7.3f} {best[2]:6d}")
print(f"{'Random projection (10 seeds)':38s} {K:6d} {rp[:,1].mean():7.3f} {rp[:,0].mean():7.3f} {'-':>6s}"
      f"   +/- {rp[:,0].std(ddof=1):.3f}")
print(f"{'Random ReLU projection (10 seeds)':38s} {K:6d} {rrp[:,1].mean():7.3f} {rrp[:,0].mean():7.3f} {'-':>6s}"
      f"   +/- {rrp[:,0].std(ddof=1):.3f}")
print(f"{'InterPLM-8M SAE (layer 4)':38s} {K:6d} {au_sae:7.3f} {ap_sae:7.3f} {nnz_sae:6d}")
print("-"*70)
print(f"\nSAE minus random ReLU projection = {ap_sae - rrp[:,0].mean():+.3f} AP")
print("  > 0 by more than the seed spread  -> learned disentanglement, §3.1 survives")
print("  ~ 0                               -> capacity + sparsity alone, §3.1 must be rewritten")

## 4. M5 — Williams' test on the phylogenetic contrasts

Felsenstein (1985) contrasts are recomputed here from scratch so that the correlation *between the two
predictors* (`r_xz`) is available — it is required by Williams' test and was never reported.

In [ ]:
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor
from scipy import stats
import numpy as np, itertools

ev = S[S.label.isin(["positive","negative"])].reset_index(drop=True)
names = list(ev.accession); n = len(names)
print(f"{n} labelled sequences -> pairwise global alignment -> NJ tree")

# Distance from GLOBAL ALIGNMENT scores, not from position-by-position comparison of
# unaligned strings. (Comparing column i of two sequences of different length is exactly
# the error that produced the retracted 'ancestral reconstruction'; see audit F2.)
al = PairwiseAligner()
al.mode = "global"
al.substitution_matrix = substitution_matrices.load("BLOSUM62")
al.open_gap_score, al.extend_gap_score = -11, -1

sq = {a: seqs[a] for a in names}
self_s = {a: al.score(sq[a], sq[a]) for a in names}
lower = [[0.0]*(i+1) for i in range(n)]
for i in range(n):
    for j in range(i):
        s = al.score(sq[names[i]], sq[names[j]])
        norm = s / np.sqrt(self_s[names[i]] * self_s[names[j]])
        lower[i][j] = float(max(1.0 - norm, 1e-6))
    if i % 25 == 0: print(f"  {i}/{n}", end="\r")

tree = DistanceTreeConstructor().nj(DistanceMatrix(names, lower))
tree.root_at_midpoint()
for cl in tree.find_clades():
    if cl.branch_length is None or cl.branch_length <= 0: cl.branch_length = 1e-6
print(f"\ntree built: {len(tree.get_terminals())} tips")

def z(d):
    v = np.array(list(d.values()), float)
    return {k: (x - v.mean())/v.std() for k, x in d.items()}

traits = {
 "lum": z(dict(zip(ev.accession, (ev.label=="positive").astype(float)))),
 "esm": z(dict(zip(ev.accession, ev["esm2_t12_35M_UR50D:cvlayer4"].astype(float)))),
 "idt": z(dict(zip(ev.accession, ev["nn_identity"].astype(float)))),
}

def contrasts(tree, vals):
    """Felsenstein (1985) standardised independent contrasts."""
    out = []
    def rec(cl):
        if cl.is_terminal():
            return vals[cl.name], (cl.branch_length or 1e-6)
        kids = [rec(c) for c in cl.clades]
        while len(kids) > 2:                      # resolve polytomy by sequential pairing
            (x1,v1),(x2,v2) = kids.pop(0), kids.pop(0)
            out.append((x1-x2)/np.sqrt(v1+v2))
            kids.insert(0, ((x1/v1 + x2/v2)/(1/v1 + 1/v2), v1*v2/(v1+v2)))
        if len(kids) == 1:
            x, v = kids[0]; return x, v + (cl.branch_length or 1e-6)
        (x1,v1),(x2,v2) = kids
        out.append((x1-x2)/np.sqrt(v1+v2))
        return ((x1/v1 + x2/v2)/(1/v1 + 1/v2),
                (cl.branch_length or 1e-6) + v1*v2/(v1+v2))
    rec(tree.root)
    return np.array(out)

c_lum, c_esm, c_idt = (contrasts(tree, traits[k]) for k in ("lum","esm","idt"))
m = np.isfinite(c_lum) & np.isfinite(c_esm) & np.isfinite(c_idt)
c_lum, c_esm, c_idt = c_lum[m], c_esm[m], c_idt[m]
N = len(c_lum)

r_xy, p_xy = stats.pearsonr(c_esm, c_lum)      # ESM-2 vs luminescence
r_xz, p_xz = stats.pearsonr(c_idt, c_lum)      # identity vs luminescence
r_yz, _    = stats.pearsonr(c_esm, c_idt)      # never reported, required by Williams

det = 1 - r_xy**2 - r_xz**2 - r_yz**2 + 2*r_xy*r_xz*r_yz
t = (r_xy - r_xz) * np.sqrt(((N-1)*(1+r_yz)) /
    (2*((N-1)/(N-3))*det + ((r_xy+r_xz)**2/4)*(1-r_yz)**3))
p_w = 2*(1 - stats.t.cdf(abs(t), N-3))

print(f"\nIndependent contrasts: N = {N}")
print(f"  PIC r (ESM-2    vs luminescence) = {r_xy:+.3f}   p = {p_xy:.3g}")
print(f"  PIC r (identity vs luminescence) = {r_xz:+.3f}   p = {p_xz:.3g}")
print(f"  PIC r (ESM-2    vs identity)     = {r_yz:+.3f}   <-- required by Williams, never reported")
print(f"\nWilliams' test (dependent correlations sharing a criterion):")
print(f"  difference = {r_xy - r_xz:+.3f},  t({N-3}) = {t:.3f},  p = {p_w:.4f}")
print("  ->", "ESM-2 contrasts significantly better (p < 0.05)" if p_w < 0.05 else
      "NOT significantly different - drop the comparative claim and say so in §2.7 and §3.4")

## 5. Reporting

Paste the two tables into the manuscript and update §3.4 Limitations items 5 and 6 to say the controls
were **run**, with the outcome, rather than that they are outstanding.

Be willing to lose both. A capacity-matched control that kills §3.1, or a Williams test that returns
p > 0.05, is a result — and reporting it is what makes the rest of the paper credible. The project's
central finding (ESM-2 beats homology on close-ancestry negatives, +0.236 AP on the strict set) does not
depend on either of these.